#BASIC

In [0]:
%sql
-- 1
CREATE CATALOG IF NOT EXISTS cyntexa_dev

In [0]:
%sql
--1
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.sales

In [0]:
%sql
-- 2
CREATE TABLE cyntexa_dev.sales.orders_raw (
    order_id INT, 
    customer_id INT,
    email STRING,
    order_date DATE, 
    total_amount DOUBLE, 
    order_status STRING, 
    payment_status STRING
) 

In [0]:
%sql
--2 
INSERT INTO cyntexa_dev.sales.orders_raw VALUES
(1001, 10, 'aarav@gmail.com', '2026-08-01', 2500.50, 'Completed', 'Paid'),
(1002, 12, 'riya@gmail.com', '2026-08-02', 1200.00, 'Completed', 'Paid'),
(1003, 13, 'karan@gmail.com', '2026-08-03', 3500.75, 'Pending', 'Pending'),
(1004, 14, 'neha@gmail.com', '2026-08-04', 850.25, 'Completed', 'Paid'),
(1005, 15, 'rahul@gmail.com', '2026-08-05', 4500.00, 'Cancelled', 'Refunded'),
(1006, 16, 'priya@gmail.com', '2026-08-06', 1750.50, 'Completed', 'Paid'),
(1007, 17, 'aman@gmail.com', '2026-08-07', 2200.00, 'Shipped', 'Paid'),
(1008, 18, 'simran@gmail.com', '2026-08-08', 999.99, 'Pending', 'Pending'),
(1009, 19, 'vikas@gmail.com', '2026-08-09', 3100.25, 'Completed', 'Paid'),
(1010, 20, 'pooja@gmail.com', '2026-08-10', 675.50, 'Shipped', 'Paid');

In [0]:
%sql
-- 3
CREATE VIEW cyntexa_dev.sales.orders_view AS SELECT * FROM cyntexa_dev.sales.orders_raw WHERE order_status = 'Completed'
    

In [0]:
%sql
-- 4a
SELECT * FROM samples.tpch.orders where
o_orderpriority = '1-URGENT' AND o_orderstatus ='F' AND o_totalprice <2000;
    


In [0]:
%sql
-- 4b
SELECT * FROM samples.tpch.customer where c_mktsegment='BUILDING' AND c_acctbal > 9000 AND c_acctbal < 9100;
    


In [0]:
%sql
-- 4c
SELECT * FROM samples.tpch.supplier where s_acctbal > 9000;

#INTERMEDIATE

In [0]:
%sql
-- 5 
CREATE FUNCTION cyntexa_dev.sales.datamask(x STRING) RETURNS STRING 
RETURN CONCAT(REPEAT('*',LENGTH(x)-4),RIGHT(x,4))

In [0]:
%sql
-- 5 
SELECT order_id, customer_id, cyntexa_dev.sales.datamask(email) as masked_email FROM cyntexa_dev.sales.orders_raw

6 - no cloud access 

but if we have a cloud access, we can create external table and define the location using keyword LOCATION "..(path)...."

In [0]:
%sql
-- 7
CREATE TABLE cyntexa_dev.sales.customers AS
SELECT
    CAST(customer_id AS BIGINT) AS customer_id,
    name,
    email,
    city,
    state,
    CAST(signup_date AS DATE) AS signup_date,
    phone 

FROM dev.bronze.customers_raw;

In [0]:
%sql
-- 7
CREATE VIEW total_spend_per_customer AS 
SELECT c.customer_id, SUM(o.total_amount) AS total_spend
FROM cyntexa_dev.sales.orders_view o
JOIN cyntexa_dev.sales.customers c
ON o.customer_id = c.customer_id 
GROUP BY c.customer_id


#ADVANCED TASKS

In [0]:
%sql
-- 8 
CREATE CATALOG IF NOT EXISTS cyntexa_dev;
CREATE CATALOG IF NOT EXISTS cyntexa_staging;
CREATE CATALOG IF NOT EXISTS cyntexa_prod;

CREATE SCHEMA IF NOT EXISTS cyntexa_dev.sales;
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.finance;
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.hr;
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.marketing;

CREATE SCHEMA IF NOT EXISTS cyntexa_staging.sales;
CREATE SCHEMA IF NOT EXISTS cyntexa_staging.finance;
CREATE SCHEMA IF NOT EXISTS cyntexa_staging.hr;
CREATE SCHEMA IF NOT EXISTS cyntexa_staging.marketing;

CREATE SCHEMA IF NOT EXISTS cyntexa_prod.sales;
CREATE SCHEMA IF NOT EXISTS cyntexa_prod.finance;
CREATE SCHEMA IF NOT EXISTS cyntexa_prod.hr;
CREATE SCHEMA IF NOT EXISTS cyntexa_prod.marketing;


-- 3 LEVEL namespace is one of the way to access the tables,functions,volumes etc. it is like catalog.schema.table,
-- it is more cleaner and easy to read.

-- dev is used for developers experiment and test
-- staging is used for final testing
-- prod is used for trusted production data



8. 
In real production there are mainly 3 catalogs - dev,uat/staging and prod

dev - this catalog is used in deveopment phase.developers used to write code , add new features, delete old features.

uat/statging - 
This catalog used for testing environment .In this the features or new updates (devloped in development phase)are tested with different test cases.if the feature pass all test cases it sends to the production else developers fix the bug or optimize it.

prod - 
in prod environment , the application or feature is open for the use for users.

9.
- Data masking strategy is the way to hide the sensative data.
-columns need to hide - credit_card, phone_no,salary and it vary according to business. 

- role tiers should see unmasked data - Admin and it also depends on the type of data that which team can see it for example - 
HR or finance team can see customer/employee team

- in unity catalog , we can define a masked function through which senstive data can be hide.
- unity catalog allow the specific group of team to see the specific data.Through unity catalog it is selected that which group/team can only read or read-write the specific data.

In [0]:
%sql
-- 10
WITH customer_revenue AS (
    SELECT 
        r.r_name AS region_name,
        c.c_custkey AS customer_id,
        c.c_name AS customer_name,
        SUM(o.o_totalprice) AS total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY r.r_name 
            ORDER BY SUM(o.o_totalprice) DESC
        ) AS revenue_rank
    FROM samples.tpch.customer c
    JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
    JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
    JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
    GROUP BY 
        r.r_name, 
        c.c_custkey, 
        c.c_name
)
SELECT 
    region_name,
    customer_id,
    customer_name,
    total_revenue,
    revenue_rank
FROM customer_revenue
WHERE revenue_rank <= 5
ORDER BY 
    region_name, 
    revenue_rank;
